# FrameWeave Hybrid v6.0 — real-frame visualizations (Ovis2.5-9B)

Chỉ inference. Không train, không đổi prompt, frame order, pixel budget hay checkpoint. Hình ghi ở `outputs_frameweave_v6_visualizations`, không ghi đè CSV eval.

Ba câu hỏi, đúng tensor của Hybrid:

1. `alpha`: question-guided pooling trong từng frame.
2. `pi` và `gamma`: cross-frame local routing. `R = gamma * pi`.
3. `eta`: relative token correction của residual thực sự được scatter vào decoder.

Survey chọn khoảng 28 câu theo task và evidence trước khi vẽ heatmap. Smoke lấy câu đầu của manifest, rồi collect chạy cả danh sách đó. `RUN_INTERVENTION = False` mặc định. CPU checks không được tính là GPU smoke. `paper_selection.json` để trống cho đến khi xem contact sheet.

## Lệnh

Các lệnh này là lệnh thật của `run_frame_visualizations.py`. Cell 7–10 gọi đúng các lệnh đó, kèm checkpoint và video lite_v2.

```bash
python run_frame_visualizations.py audit   --root /content   --output-dir /content/outputs_frameweave_v6_visualizations   --adapter-checkpoint /content/outputs_reljoin_v6_ovis/checkpoints/best_external_val/Hybrid.pt   --model-dir /content/checkpoints/ovis2_5_9b   --herbench-root /content/HERBench_official   --videos-root /content/HERBench_official/videos   --herbench-revision 30b42e97180f2c7c0fdc3f2aed412ae04f0366ca   --model-revision 5a1f9b995c98ae5b85a28ec89d88d74295be0ea7   --herbench-config lite_v2

python run_frame_visualizations.py cpu-checks   --root /content   --output-dir /content/outputs_frameweave_v6_visualizations

python run_frame_visualizations.py survey   --root /content   --output-dir /content/outputs_frameweave_v6_visualizations   --herbench-root /content/HERBench_official   --videos-root /content/HERBench_official/videos   --herbench-revision 30b42e97180f2c7c0fdc3f2aed412ae04f0366ca   --herbench-config lite_v2

python run_frame_visualizations.py smoke   --root /content   --output-dir /content/outputs_frameweave_v6_visualizations   --selection-manifest /content/outputs_frameweave_v6_visualizations/survey_manifest.json   --adapter-checkpoint /content/outputs_reljoin_v6_ovis/checkpoints/best_external_val/Hybrid.pt   --model-dir /content/checkpoints/ovis2_5_9b   --herbench-root /content/HERBench_official   --videos-root /content/HERBench_official/videos   --herbench-base-vlm /content/herbench_eval_pins/base_vlm.py   --herbench-base-sha256 6104884c366906d5c223cc9727c3bad778dcdf98c881c4b17b556c20e940af97   --herbench-revision 30b42e97180f2c7c0fdc3f2aed412ae04f0366ca   --model-revision 5a1f9b995c98ae5b85a28ec89d88d74295be0ea7   --herbench-config lite_v2   --query-method top_eta   --max-queries 8   --limit 1

python run_frame_visualizations.py collect   --root /content   --output-dir /content/outputs_frameweave_v6_visualizations   --selection-manifest /content/outputs_frameweave_v6_visualizations/survey_manifest.json   --adapter-checkpoint /content/outputs_reljoin_v6_ovis/checkpoints/best_external_val/Hybrid.pt   --model-dir /content/checkpoints/ovis2_5_9b   --herbench-root /content/HERBench_official   --videos-root /content/HERBench_official/videos   --herbench-base-vlm /content/herbench_eval_pins/base_vlm.py   --herbench-base-sha256 6104884c366906d5c223cc9727c3bad778dcdf98c881c4b17b556c20e940af97   --herbench-revision 30b42e97180f2c7c0fdc3f2aed412ae04f0366ca   --model-revision 5a1f9b995c98ae5b85a28ec89d88d74295be0ea7   --herbench-config lite_v2   --query-method top_eta   --max-queries 8

python run_frame_visualizations.py render   --root /content   --output-dir /content/outputs_frameweave_v6_visualizations

python run_frame_visualizations.py contact-sheet   --root /content   --output-dir /content/outputs_frameweave_v6_visualizations

python run_frame_visualizations.py intervene   --root /content   --output-dir /content/outputs_frameweave_v6_visualizations   --adapter-checkpoint /content/outputs_reljoin_v6_ovis/checkpoints/best_external_val/Hybrid.pt   --model-dir /content/checkpoints/ovis2_5_9b   --herbench-root /content/HERBench_official   --videos-root /content/HERBench_official/videos   --herbench-base-vlm /content/herbench_eval_pins/base_vlm.py   --herbench-base-sha256 6104884c366906d5c223cc9727c3bad778dcdf98c881c4b17b556c20e940af97   --herbench-revision 30b42e97180f2c7c0fdc3f2aed412ae04f0366ca   --model-revision 5a1f9b995c98ae5b85a28ec89d88d74295be0ea7   --herbench-config lite_v2
```

`intervene` chỉ chạy khi cell 10 thấy `RUN_INTERVENTION = True`.

## Source code

Mở notebook này từ GitHub hoặc Colab. Run All clone [FrameWeave](https://github.com/nqMinh-GR/FrameWeave.git) vào `/content/FrameWeave`, rồi symlink `reljoin/` và `run_*.py` vào `/content`. Không upload file source.

Checkpoint v3/v4/v5 không được load. Colab **A100 40 GB**. Có thể tái sử dụng `ovis_env`, Ovis2.5-9B và video lite_v2 đã tải cho notebook eval chính.



## 1. Configuration


In [ ]:
# 1. Frame-visualization config — HERBench lite_v2 / fixed Hybrid.pt
from __future__ import annotations

from pathlib import Path

assert Path("/content").exists(), "Notebook này chỉ chạy trên Google Colab (/content)."

ROOT = Path("/content")
SMOKE_FIRST = True
RUN_BATCH = True
BATCH_N = 24  # không còn lấy 24 câu đầu. Survey manifest quyết định danh sách chạy.
RUN_INTERVENTION = False
QUERY_METHOD = "top_eta"   # top_eta | explicit | roi
MAX_QUERIES = 8
EXAMPLE_IDS = ""           # để trống: dùng survey_manifest.json
METHOD_VERSION = "6.0.0"

VARIANT_TO_METHOD = {
    "Hybrid": "hybrid",
}
REQUIRED_CKPTS = ["Hybrid.pt"]

EXTRACT_DIR_CANDIDATES = [
    ROOT / "outputs_reljoin_v6_ovis",
    ROOT / "outputs_reljoin_v6_ovis (2)",
    Path("/content/drive/MyDrive/outputs_reljoin_v6_ovis"),
    Path("/content/drive/MyDrive/reljoin_v6_ovis_outputs") / "outputs_reljoin_v6_ovis",
    Path("/content/drive/MyDrive/reljoin_v6_ovis_outputs"),
]

MODEL_ID = "ATH-MaaS/Ovis2.5-9B"
MODEL_REVISION = "5a1f9b995c98ae5b85a28ec89d88d74295be0ea7"
HERBENCH_ID = "DanBenAmi/HERBench"
HERBENCH_REVISION = "30b42e97180f2c7c0fdc3f2aed412ae04f0366ca"
HERBENCH_BASE_SHA256 = "6104884c366906d5c223cc9727c3bad778dcdf98c881c4b17b556c20e940af97"
HERBENCH_ACC_SHA256 = "eb67b618f3004b52c8290b2fd30e4012153a66f3f93d576c29760280caa3344d"
HERBENCH_BASE_URL = "https://raw.githubusercontent.com/DanBenAmi/HERBench/main/evaluation/model_wrappers/base_vlm.py"
HERBENCH_ACC_URL = "https://raw.githubusercontent.com/DanBenAmi/HERBench/main/evaluation/calculate_accuracy.py"

VENV = ROOT / "ovis_env"
PYTHON = VENV / "bin" / "python"
OUT_DIR = ROOT / "outputs_reljoin_v6_ovis"
CKPT_DIR = OUT_DIR / "checkpoints" / "best_external_val"
VIZ_DIR = ROOT / "outputs_frameweave_v6_visualizations"
MODEL_DIR = ROOT / "checkpoints" / "ovis2_5_9b"
HERBENCH_ROOT = ROOT / "HERBench_official"
HERBENCH_VIDEOS = HERBENCH_ROOT / "videos"
EVAL_DIR = ROOT / "herbench_eval_pins"

print({
    "SMOKE_FIRST": SMOKE_FIRST,
    "RUN_BATCH": RUN_BATCH,
    "BATCH_N": BATCH_N,
    "RUN_INTERVENTION": RUN_INTERVENTION,
    "QUERY_METHOD": QUERY_METHOD,
    "MAX_QUERIES": MAX_QUERIES,
    "EXAMPLE_IDS": EXAMPLE_IDS,
    "VIZ_DIR": str(VIZ_DIR),
    "REQUIRED_CKPTS": REQUIRED_CKPTS,
    "EXTRACT_DIR_CANDIDATES": [str(p) for p in EXTRACT_DIR_CANDIDATES],
    "CKPT_DIR": str(CKPT_DIR),
    "MODEL_DIR": str(MODEL_DIR),
    "method": "FrameWeave-Hybrid",
    "method_version": METHOD_VERSION,
    "backbone": MODEL_ID,
})


## 2. Environment (`ovis_env`)

Official Ovis2.5-9B pins: `torch==2.4.0`, `transformers==4.51.3`, `flash-attn==2.7.0.post2`.
Stop if flash-attn install fails — the ViT hard-imports it.


In [ ]:
# 2. Bootstrap isolated ovis_env (Python 3.11 + torch 2.4.0 + flash-attn 2.7)
from __future__ import annotations

import os
import subprocess
from pathlib import Path

assert Path("/content").exists(), "This notebook targets Google Colab (/content)."
ROOT = Path("/content")
VENV = ROOT / "ovis_env"
VENV_BIN = VENV / "bin"
PYTHON = VENV_BIN / "python"
PIP = VENV_BIN / "pip"
LOG = ROOT / "outputs_reljoin_v6_ovis" / "logs" / "cell2_env.log"
LOG.parent.mkdir(parents=True, exist_ok=True)
LOG.write_text("", encoding="utf-8")
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"


def sh(cmd, check=True, env=None):
    line = "$ " + " ".join(map(str, cmd))
    print(line, flush=True)
    with LOG.open("a", encoding="utf-8") as f:
        f.write(line + "\n")
        proc = subprocess.run(
            list(map(str, cmd)),
            check=False,
            text=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            env=env,
        )
        f.write(proc.stdout or "")
        f.write(f"\nexit={proc.returncode}\n\n")
    print(proc.stdout or "", end="")
    if check and proc.returncode != 0:
        print("LOG:", LOG)
        print((proc.stdout or "")[-4000:])
        raise RuntimeError(f"command failed ({proc.returncode}). See {LOG}")
    return proc


sh(["apt-get", "update", "-qq"], check=False)
sh(["apt-get", "install", "-y", "-qq", "python3.11", "python3.11-venv", "python3.11-dev", "ffmpeg"], check=False)
py311 = Path("/usr/bin/python3.11")
assert py311.exists(), "Không tìm thấy python3.11 — kiểm tra apt install."

need_recreate = True
if PYTHON.exists():
    ver = subprocess.check_output([str(PYTHON), "-V"], text=True).strip()
    print("Existing venv:", ver)
    need_recreate = "3.11" not in ver
if need_recreate:
    if VENV.exists():
        sh(["rm", "-rf", str(VENV)])
    sh([str(py311), "-m", "venv", "--copies", str(VENV)])
if not PIP.exists():
    sh([str(PYTHON), "-m", "ensurepip", "--upgrade"], check=False)
if not PIP.exists():
    get_pip = Path("/tmp/get-pip.py")
    sh(["curl", "-sSL", "https://bootstrap.pypa.io/get-pip.py", "-o", str(get_pip)])
    sh([str(PYTHON), str(get_pip)])
sh([str(PYTHON), "-m", "pip", "install", "-q", "--upgrade", "pip", "setuptools<70", "wheel"])
print("Python in venv:", subprocess.check_output([str(PYTHON), "-V"], text=True).strip())
sh([
    str(PYTHON), "-m", "pip", "install", "--disable-pip-version-check",
    "torch==2.4.0", "torchvision==0.19.0",
    "--index-url", "https://download.pytorch.org/whl/cu121",
])
sh([
    str(PYTHON), "-m", "pip", "install", "--disable-pip-version-check", "--upgrade-strategy", "only-if-needed",
    "transformers==4.51.3", "numpy==1.25.0", "pillow==10.3.0", "moviepy==1.0.3",
    "huggingface_hub", "hf-transfer", "datasets", "accelerate", "einops", "sentencepiece",
    "opencv-python-headless", "pandas", "scipy", "tqdm", "ninja", "packaging",
])
print("Installing flash-attn==2.7.0.post2 (required by Ovis ViT)...")
fa_env = os.environ.copy()
fa_env["MAX_JOBS"] = "4"
sh(
    [str(PYTHON), "-m", "pip", "install", "flash-attn==2.7.0.post2", "--no-build-isolation"],
    env=fa_env,
)
check = subprocess.check_output([
    str(PYTHON), "-c",
    "import torch, transformers, flash_attn; "
    "print(torch.__version__); print(transformers.__version__); print(flash_attn.__version__); "
    "print(torch.__file__); print(transformers.__file__)",
], text=True)
print(check)
for line in check.splitlines():
    if "/site-packages/" in line:
        assert "/content/ovis_env/" in line, f"Package leaked outside venv: {line}"
print("ovis_env ready:", PYTHON)
print("LOG:", LOG)


## 3. Drive + pin HERBench evaluators

Không tải lại evaluator nếu SHA đã khớp (tránh GitHub 429). Không clone DTR.


In [ ]:
# 3. Drive + pin HERBench evaluator (skip download if SHA matches)
from __future__ import annotations

import hashlib
import os
import time
import urllib.request
from pathlib import Path

ROOT = Path("/content")
VENV = ROOT / "ovis_env"
PYTHON = VENV / "bin" / "python"
HERBENCH_ROOT = ROOT / "HERBench_official"
HERBENCH_VIDEOS = HERBENCH_ROOT / "videos"
MODEL_DIR = ROOT / "checkpoints" / "ovis2_5_9b"
OUT_DIR = ROOT / "outputs_reljoin_v6_ovis"
EVAL_DIR = ROOT / "herbench_eval_pins"
HERBENCH_BASE_URL = "https://raw.githubusercontent.com/DanBenAmi/HERBench/main/evaluation/model_wrappers/base_vlm.py"
HERBENCH_ACC_URL = "https://raw.githubusercontent.com/DanBenAmi/HERBench/main/evaluation/calculate_accuracy.py"
HERBENCH_BASE_SHA256 = "6104884c366906d5c223cc9727c3bad778dcdf98c881c4b17b556c20e940af97"
HERBENCH_ACC_SHA256 = "eb67b618f3004b52c8290b2fd30e4012153a66f3f93d576c29760280caa3344d"
JSDELIVR_BASE = "https://cdn.jsdelivr.net/gh/DanBenAmi/HERBench@main/evaluation/model_wrappers/base_vlm.py"
JSDELIVR_ACC = "https://cdn.jsdelivr.net/gh/DanBenAmi/HERBench@main/evaluation/calculate_accuracy.py"

for p in [HERBENCH_ROOT, HERBENCH_VIDEOS, MODEL_DIR.parent, EVAL_DIR, OUT_DIR, OUT_DIR / "predictions_residual_direction"]:
    p.mkdir(parents=True, exist_ok=True)

try:
    from google.colab import drive

    drive.mount("/content/drive", force_remount=False)
    RELJOIN_PERSIST = Path("/content/drive/MyDrive/reljoin_v6_ovis_outputs")
except Exception as exc:
    print("Drive mount skipped:", exc)
    RELJOIN_PERSIST = OUT_DIR / "persist_local"
if RELJOIN_PERSIST.resolve() == OUT_DIR.resolve():
    RELJOIN_PERSIST = OUT_DIR / "persist_local"
    print("Persist dir was out_dir; using", RELJOIN_PERSIST)
RELJOIN_PERSIST.mkdir(parents=True, exist_ok=True)
os.environ["RELJOIN_PERSIST"] = str(RELJOIN_PERSIST)
os.environ["RELJOIN_ROOT"] = str(ROOT)


def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def fetch_pinned(urls: list[str], dest: Path, expected: str) -> None:
    if dest.exists() and sha256_file(dest) == expected:
        print("Already pinned:", dest.name)
        return
    last_err = None
    for url in urls:
        for attempt in range(4):
            try:
                print(f"GET {url} (try {attempt + 1})")
                urllib.request.urlretrieve(url, dest)
                got = sha256_file(dest)
                if got == expected:
                    print("OK", dest.name)
                    return
                last_err = RuntimeError(f"SHA mismatch for {dest.name}: {got}")
            except Exception as exc:
                last_err = exc
                time.sleep(2 ** attempt)
    raise RuntimeError(f"Cannot pin {dest.name}: {last_err}")


fetch_pinned([HERBENCH_BASE_URL, JSDELIVR_BASE], EVAL_DIR / "base_vlm.py", HERBENCH_BASE_SHA256)
fetch_pinned([HERBENCH_ACC_URL, JSDELIVR_ACC], EVAL_DIR / "calculate_accuracy.py", HERBENCH_ACC_SHA256)

print({
    "RELJOIN_PERSIST": str(RELJOIN_PERSIST),
    "OUT_DIR": str(OUT_DIR),
    "MODEL_DIR": str(MODEL_DIR),
    "EVAL_DIR": str(EVAL_DIR),
})


## 4. Clone source and verify


In [ ]:
# Fetch FrameWeave source. Run All clones the repo; do not upload .py files.
from __future__ import annotations

import subprocess
from pathlib import Path

REPO_URL = "https://github.com/nqMinh-GR/FrameWeave.git"
REPO_DIR = Path("/content/FrameWeave")
ROOT = Path("/content")
assert ROOT.exists(), "This notebook targets Google Colab (/content)."

if not (REPO_DIR / ".git").exists():
    subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)])
else:
    subprocess.check_call(["git", "-C", str(REPO_DIR), "pull", "--ff-only"])


def link(src: Path, dest: Path) -> None:
    src = src.resolve()
    if dest.is_symlink():
        if dest.resolve() != src:
            dest.unlink()
            dest.symlink_to(src)
        return
    if dest.exists():
        raise RuntimeError(
            f"{dest} already exists and is not the repo link. Restart the runtime, then Run All."
        )
    dest.symlink_to(src)


CODE = REPO_DIR / "ovis"
link(CODE / "reljoin", ROOT / "reljoin")
for script in sorted(CODE.glob("run_*.py")):
    link(script, ROOT / script.name)
print("FrameWeave source linked from", CODE)


In [ ]:
# 4. Verify frame-visualization sources
from pathlib import Path

ROOT = Path("/content")
required = [
    ROOT / "run_frame_visualizations.py",
    ROOT / "run_herbench_oracle_reljoin.py",
    ROOT / "reljoin" / "__init__.py",
    ROOT / "reljoin" / "config.py",
    ROOT / "reljoin" / "train.py",
    ROOT / "reljoin" / "reljoin_modules.py",
    ROOT / "reljoin" / "ovis_hooks.py",
    ROOT / "reljoin" / "ovis_generate.py",
    ROOT / "reljoin" / "herbench_oracle.py",
    ROOT / "reljoin" / "diagnostics.py",
    ROOT / "reljoin" / "losses.py",
    ROOT / "reljoin" / "frame_viz" / "__init__.py",
    ROOT / "reljoin" / "frame_viz" / "collector.py",
    ROOT / "reljoin" / "frame_viz" / "geometry.py",
    ROOT / "reljoin" / "frame_viz" / "render.py",
    ROOT / "reljoin" / "frame_viz" / "pipeline.py",
    ROOT / "reljoin" / "frame_viz" / "survey.py",
    ROOT / "reljoin" / "frame_viz" / "cpu_checks.py",
]
missing = [str(p) for p in required if not p.exists()]
assert not missing, "Missing files — clone or symlink failed:\n" + "\n".join(missing)
print("Frame-visualization sources present.")
for p in required:
    print(f"  OK {p} ({p.stat().st_size} bytes)")


## 5. Gắn checkpoint (`outputs_reljoin_v6_ovis/checkpoints`)

Cùng layout train lite joint. Bắt buộc `Hybrid.pt` (`VARIANT=Hybrid`, `TRAIN_STAGE=joint`). Checkpoint v3/v4/v5 **không** load vào v6.0.


In [ ]:
# 5. Bind extract checkpoints — same layout as train.py / make_output_tree
from __future__ import annotations

import json
import shutil
import zipfile
from pathlib import Path

ROOT = Path("/content")
OUT_DIR = ROOT / "outputs_reljoin_v6_ovis"
OUT_DIR.mkdir(parents=True, exist_ok=True)
(OUT_DIR / "predictions_residual_direction").mkdir(parents=True, exist_ok=True)

try:
    REQUIRED_CKPTS
    EXTRACT_DIR_CANDIDATES
    VARIANT_TO_METHOD
except NameError:
    REQUIRED_CKPTS = ["Hybrid.pt"]
    VARIANT_TO_METHOD = {
        "Hybrid": "hybrid",
    }
    EXTRACT_DIR_CANDIDATES = [
        ROOT / "outputs_reljoin_v6_ovis",
        ROOT / "outputs_reljoin_v6_ovis (2)",
        Path("/content/drive/MyDrive/outputs_reljoin_v6_ovis"),
        Path("/content/drive/MyDrive/reljoin_v6_ovis_outputs") / "outputs_reljoin_v6_ovis",
        Path("/content/drive/MyDrive/reljoin_v6_ovis_outputs"),
    ]

v2_pt = ROOT / "outputs_reljoin_residual" / "checkpoints" / "best_external_val" / "RelJoin.pt"
if v2_pt.is_file():
    print("NOTE: found Ovis v2 RelJoin.pt at", v2_pt, "— không dùng cho v6.0")


def ckpt_complete(best_dir: Path) -> bool:
    return best_dir.is_dir() and all((best_dir / n).is_file() for n in REQUIRED_CKPTS)


def find_extract_root() -> Path:
    for cand in EXTRACT_DIR_CANDIDATES:
        best = cand / "checkpoints" / "best_external_val"
        if ckpt_complete(best):
            return cand
        last = cand / "checkpoints" / "last"
        if ckpt_complete(last):
            print("WARN: using checkpoints/last (best_external_val incomplete) at", cand)
            return cand
        if (cand / "Hybrid.pt").is_file():
            return cand
    zip_roots = [
        ROOT,
        Path("/content/drive/MyDrive"),
        Path("/content/drive/MyDrive/reljoin_v6_ovis_outputs"),
    ]
    for base in zip_roots:
        if not base.exists():
            continue
        for zpath in list(base.glob("*reljoin*.zip")) + list(base.glob("*output*.zip")):
            if not zpath.is_file():
                continue
            dest = Path("/tmp/vast_outputs_reljoin_v6_ovis")
            if dest.exists():
                shutil.rmtree(dest)
            print("Extracting", zpath)
            with zipfile.ZipFile(zpath) as z:
                z.extractall(dest)
            for cand in [dest, *dest.iterdir()]:
                if cand.is_dir() and ckpt_complete(cand / "checkpoints" / "best_external_val"):
                    return cand
                if cand.is_dir() and ckpt_complete(cand / "checkpoints" / "last"):
                    print("WARN: zip extract using checkpoints/last at", cand)
                    return cand
    scanned = "\n".join(
        f"  {c} exists={c.exists()} best={list((c / 'checkpoints' / 'best_external_val').glob('*.pt')) if (c / 'checkpoints' / 'best_external_val').exists() else 'NO DIR'}"
        for c in EXTRACT_DIR_CANDIDATES
    )
    raise FileNotFoundError(
        "Không thấy Hybrid.pt đúng layout train:\n"
        "  {extract}/checkpoints/best_external_val/Hybrid.pt\n"
        "Upload cả folder `outputs_reljoin_v6_ovis` (kèm checkpoints/) lên /content hoặc Drive.\n"
        f"Đã quét:\n{scanned}"
    )


EXTRACT_DIR = find_extract_root()
BEST_DIR = EXTRACT_DIR / "checkpoints" / "best_external_val"
LAST_DIR = EXTRACT_DIR / "checkpoints" / "last"
if ckpt_complete(BEST_DIR):
    CKPT_DIR = BEST_DIR
elif ckpt_complete(LAST_DIR):
    CKPT_DIR = LAST_DIR
else:
    reljoin_loose = EXTRACT_DIR / "Hybrid.pt"
    CKPT_DIR = EXTRACT_DIR if reljoin_loose.is_file() else BEST_DIR
CKPT_DIR.mkdir(parents=True, exist_ok=True)

resolved = {}
for name in REQUIRED_CKPTS:
    src = CKPT_DIR / name
    if not src.is_file():
        alt = LAST_DIR / name
        if alt.is_file():
            print(f"best missing {name} → copy from last/")
            shutil.copy2(alt, CKPT_DIR / name)
            src = CKPT_DIR / name
        else:
            resolved[name] = None
            continue
    mb = src.stat().st_size / 1e6
    print(f"OK {name}: {src} ({mb:.1f} MB)")
    resolved[name] = src

missing = [n for n, p in resolved.items() if p is None]
assert not missing, f"Thiếu checkpoint: {missing} dưới {CKPT_DIR} và {LAST_DIR}"

if CKPT_DIR.resolve() != LAST_DIR.resolve():
    for variant in VARIANT_TO_METHOD:
        name = f"{variant}.pt"
        if not (CKPT_DIR / name).is_file() and (LAST_DIR / name).is_file():
            print(f"missing {name} → copy from last/")
            shutil.copy2(LAST_DIR / name, CKPT_DIR / name)

found_methods = ["vanilla"]
for variant, method in VARIANT_TO_METHOD.items():
    path = CKPT_DIR / f"{variant}.pt"
    if path.is_file():
        if path.name not in REQUIRED_CKPTS:
            mb = path.stat().st_size / 1e6
            print(f"OK {path.name}: {path} ({mb:.1f} MB)")
        found_methods.append(method)
METHODS = ",".join(found_methods)

for aux in ["train_summary.json", "config.json", "environment.txt"]:
    src = EXTRACT_DIR / aux
    if src.exists() and src.resolve() != (OUT_DIR / aux).resolve():
        shutil.copy2(src, OUT_DIR / aux)
        print("Copied", aux)
log_src = EXTRACT_DIR / "logs" / "train_metrics.csv"
if log_src.exists():
    (OUT_DIR / "logs").mkdir(parents=True, exist_ok=True)
    dst = OUT_DIR / "logs" / "train_metrics.csv"
    if log_src.resolve() != dst.resolve():
        shutil.copy2(log_src, dst)

summary_path = EXTRACT_DIR / "train_summary.json"
if summary_path.exists():
    print("train_summary:", json.loads(summary_path.read_text()))

print({
    "EXTRACT_DIR": str(EXTRACT_DIR),
    "CKPT_DIR": str(CKPT_DIR),
    "METHODS": METHODS,
    "pts": sorted(p.name for p in CKPT_DIR.glob("*.pt")),
})


## 6. Tải Ovis2.5-9B + HERBench lite videos (~35 GB)

PerceptionTest **không** tải — không train. Cell này bắt buộc trước smoke (`model_config` phải `True`).


In [ ]:
# 6. Download Ovis2.5-9B + HERBench lite (eval only)
from __future__ import annotations

import os
import shutil
import subprocess
from pathlib import Path

ROOT = Path("/content")
PYTHON = ROOT / "ovis_env" / "bin" / "python"
assert PYTHON.exists(), "Chạy cell 2 trước"
os.environ["PATH"] = f"{ROOT / 'ovis_env' / 'bin'}:{os.environ.get('PATH', '')}"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

HERBENCH_ROOT = ROOT / "HERBench_official"
HERBENCH_VIDEOS = HERBENCH_ROOT / "videos"
MODEL_DIR = ROOT / "checkpoints" / "ovis2_5_9b"
HERBENCH_HF_ID = "DanBenAmi/HERBench"
HERBENCH_REVISION = "30b42e97180f2c7c0fdc3f2aed412ae04f0366ca"
MODEL_HF_ID = "ATH-MaaS/Ovis2.5-9B"
MODEL_REVISION = "5a1f9b995c98ae5b85a28ec89d88d74295be0ea7"

for p in [HERBENCH_ROOT, HERBENCH_VIDEOS, MODEL_DIR.parent]:
    p.mkdir(parents=True, exist_ok=True)

print("Downloading Ovis2.5-9B (~18 GB bf16)...")
subprocess.run([str(PYTHON), "-c", f"""
import os
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
from huggingface_hub import snapshot_download
snapshot_download(
    repo_id='{MODEL_HF_ID}',
    revision='{MODEL_REVISION}',
    local_dir=r'{MODEL_DIR}',
    ignore_patterns=['*.msgpack', '*.h5', 'flax_model*', 'tf_model*'],
)
print('model OK')
"""], check=True)
assert (MODEL_DIR / "config.json").exists(), f"Missing {MODEL_DIR / 'config.json'}"

print("Downloading HERBench lite (~35GB)...")
subprocess.run([str(PYTHON), "-c", f"""
import os
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
from huggingface_hub import snapshot_download
snapshot_download(
    repo_id='{HERBENCH_HF_ID}',
    repo_type='dataset',
    revision='{HERBENCH_REVISION}',
    local_dir=r'{HERBENCH_ROOT}',
    allow_patterns=[
        'data/herbench_lite_v2.parquet',
        'data/herbench_annotations_lite_v2.json',
        'data/*metadata.json',
        'videos/videos.tar.part.00',
        'videos/videos.tar.part.01',
        'videos/videos.tar.part.02',
        'videos/videos.tar.part.03',
        'videos/videos_lite_info.txt',
        'videos/videos.tar.checksums.txt',
    ],
)
print('herbench snapshot OK')
"""], check=True)

LITE_MP4_MIN = 68
parts = sorted(HERBENCH_VIDEOS.glob("videos.tar.part.*"))
print("parts:", [p.name for p in parts])
mp4s = list(HERBENCH_VIDEOS.rglob("*.mp4"))
assert len(parts) >= 4 or len(mp4s) >= 2, (
    f"Thiếu videos.tar.part.00–03 trong {HERBENCH_VIDEOS}. Có: {[p.name for p in parts]}"
)
if len(mp4s) < LITE_MP4_MIN:
    assert len(parts) >= 4, (
        f"Chỉ có {len(mp4s)} mp4 và thiếu tar parts — tải lại videos.tar.part.00–03."
    )
    cat = subprocess.Popen(["cat", *map(str, parts)], stdout=subprocess.PIPE)
    subprocess.run(["tar", "-xf", "-", "-C", str(HERBENCH_VIDEOS)], stdin=cat.stdout, check=True)
    cat.wait()
    print("Extracted archives")
else:
    print(f"Already have {len(mp4s)} mp4")

n_mp4 = len(list(HERBENCH_VIDEOS.rglob("*.mp4")))
if n_mp4 >= LITE_MP4_MIN:
    for part in parts:
        part.unlink(missing_ok=True)
    print("Removed archives after full extract")
else:
    print(f"KEEP tar parts — only {n_mp4} mp4 (need ~{LITE_MP4_MIN} for full eval)")

print({
    "videos_mp4": n_mp4,
    "model_config": (MODEL_DIR / "config.json").exists(),
    "free_GiB": round(shutil.disk_usage(ROOT).free / 2**30, 2),
})
assert (MODEL_DIR / "config.json").exists(), "Ovis config.json missing — tải lại cell 6."
assert n_mp4 >= 2, "Lite extract xong vẫn thiếu mp4"
if n_mp4 < LITE_MP4_MIN:
    print(f"WARN: {n_mp4} mp4 < {LITE_MP4_MIN}. Smoke có thể chạy; full eval sẽ thiếu video.")
print("Downloads OK")


## 7. Audit và CPU checks

Chạy trong `ovis_env`. Không load Ovis. `validation_report.json` ghi `gpu_smoke = not_run` cho đến khi smoke thật sự xong.


In [ ]:
# 7. Audit + CPU checks. Matplotlib is only needed for figures.
from __future__ import annotations

import os
import subprocess
from pathlib import Path

ROOT = Path("/content")
PYTHON = ROOT / "ovis_env" / "bin" / "python"
PIP = ROOT / "ovis_env" / "bin" / "pip"
RUNNER = ROOT / "run_frame_visualizations.py"
VIZ_DIR = ROOT / "outputs_frameweave_v6_visualizations"
VIZ_DIR.mkdir(parents=True, exist_ok=True)
CKPT = CKPT_DIR if "CKPT_DIR" in dir() else (ROOT / "outputs_reljoin_v6_ovis" / "checkpoints" / "best_external_val")

assert PYTHON.exists(), "Chạy cell 2 trước — ovis_env missing."
assert RUNNER.is_file(), f"Missing {RUNNER}"

env = os.environ.copy()
env.pop("PYTHONPATH", None)
env.pop("PYTHONHOME", None)
env["PYTHONNOUSERSITE"] = "1"
probe = subprocess.run(
    [str(PYTHON), "-c", "import matplotlib, PIL"],
    check=False,
    env=env,
)
if probe.returncode != 0:
    subprocess.run([str(PIP), "install", "-q", "matplotlib"], check=True, env=env)

def _from_cell1(name, default):
    try:
        ip = get_ipython()
    except NameError:
        ip = None
    ns = ip.user_ns if ip is not None else {}
    return ns[name] if name in ns else default

common = [
    "--root", "/content",
    "--output-dir", str(VIZ_DIR),
    "--adapter-checkpoint", str(Path(CKPT) / "Hybrid.pt"),
    "--model-dir", str(_from_cell1("MODEL_DIR", ROOT / "checkpoints" / "ovis2_5_9b")),
    "--herbench-root", str(_from_cell1("HERBENCH_ROOT", ROOT / "HERBench_official")),
    "--videos-root", str(_from_cell1("HERBENCH_VIDEOS", ROOT / "HERBench_official" / "videos")),
    "--herbench-revision", "30b42e97180f2c7c0fdc3f2aed412ae04f0366ca",
    "--model-revision", "5a1f9b995c98ae5b85a28ec89d88d74295be0ea7",
    "--herbench-config", "lite_v2",
]
for command in ("audit", "cpu-checks"):
    cmd = [str(PYTHON), "-u", str(RUNNER), command, *common]
    print("$", " ".join(cmd), flush=True)
    subprocess.run(cmd, check=True, env=env, cwd="/content")
print("Audit and CPU checks finished. GPU smoke is still not_run.")


## 8. Survey, smoke một câu, rồi cả danh sách khảo sát

Survey không load Ovis. Nó ghi `survey_candidates.csv` cho mọi câu có video và `survey_manifest.json` cho lượt chạy. Smoke lấy câu đầu của manifest. Collect chạy cả manifest, cùng oracle frames và prompt evaluation. Collector tắt thì forward không copy `pi`. Khi bật, hai lượt scalar và `pi` phải khớp prediction và updated visual tokens trong atol đã ghi.

Nếu job còn treo: `!pkill -f run_frame_visualizations` rồi xóa `/content/frame_viz.job.json`.


In [ ]:
# 8. Survey, then smoke the first manifest example, then collect that list. Re-run this cell to reattach.
from __future__ import annotations

import json
import os
import shlex
import signal
import subprocess
import time
from pathlib import Path

ROOT = Path("/content")
PYTHON = ROOT / "ovis_env" / "bin" / "python"
RUNNER = ROOT / "run_frame_visualizations.py"
VENV = ROOT / "ovis_env"
HERBENCH_ROOT = ROOT / "HERBench_official"
HERBENCH_VIDEOS = HERBENCH_ROOT / "videos"
MODEL_DIR = ROOT / "checkpoints" / "ovis2_5_9b"
OUT_DIR = ROOT / "outputs_reljoin_v6_ovis"
VIZ_DIR = ROOT / "outputs_frameweave_v6_visualizations"
EVAL_DIR = ROOT / "herbench_eval_pins"
CKPT = CKPT_DIR if "CKPT_DIR" in dir() else (OUT_DIR / "checkpoints" / "best_external_val")
VIZ_DIR.mkdir(parents=True, exist_ok=True)
JOB_FILE = ROOT / "frame_viz.job.json"
DRIVE_PARENT = Path("/content/drive/MyDrive/reljoin_v6_ovis_outputs")
DRIVE_DIR = DRIVE_PARENT / "frameweave_v6_visualizations"

def _from_cell1(name, default):
    try:
        ip = get_ipython()
    except NameError:
        ip = None
    ns = ip.user_ns if ip is not None else {}
    return ns[name] if name in ns else default

SMOKE_FIRST = bool(_from_cell1("SMOKE_FIRST", True))
RUN_BATCH = bool(_from_cell1("RUN_BATCH", True))
BATCH_N = int(_from_cell1("BATCH_N", 24))
QUERY_METHOD = str(_from_cell1("QUERY_METHOD", "top_eta"))
MAX_QUERIES = int(_from_cell1("MAX_QUERIES", 8))
EXAMPLE_IDS = str(_from_cell1("EXAMPLE_IDS", "") or "")

HERBENCH_REVISION = "30b42e97180f2c7c0fdc3f2aed412ae04f0366ca"
MODEL_REVISION = "5a1f9b995c98ae5b85a28ec89d88d74295be0ea7"
HERBENCH_BASE_SHA256 = "6104884c366906d5c223cc9727c3bad778dcdf98c881c4b17b556c20e940af97"

needed = list(REQUIRED_CKPTS) if "REQUIRED_CKPTS" in dir() else ["Hybrid.pt"]
missing = [n for n in needed if not (CKPT / n).exists()]
assert not missing, (
    f"Chạy cell 5 trước. Thiếu {missing} trong {CKPT}. "
    "Upload folder `outputs_reljoin_v6_ovis/checkpoints/best_external_val/` lên /content."
)
assert (MODEL_DIR / "config.json").exists(), f"Chạy cell 6 trước — missing {MODEL_DIR / 'config.json'}"
assert PYTHON.exists(), "Chạy cell 2 trước — ovis_env missing."
assert RUNNER.is_file(), f"Missing {RUNNER}"
n_mp4 = len(list(HERBENCH_VIDEOS.rglob("*.mp4"))) if HERBENCH_VIDEOS.exists() else 0
print({"Hybrid.pt": str(CKPT / "Hybrid.pt"), "videos_mp4": n_mp4, "VIZ_DIR": str(VIZ_DIR)})
assert n_mp4 >= 1, f"Chạy cell 6 trước — không có mp4 trong {HERBENCH_VIDEOS}."

common = [
    "--root", "/content",
    "--herbench-root", str(HERBENCH_ROOT),
    "--videos-root", str(HERBENCH_VIDEOS),
    "--model-dir", str(MODEL_DIR),
    "--adapter-checkpoint", str(CKPT / "Hybrid.pt"),
    "--output-dir", str(VIZ_DIR),
    "--herbench-revision", HERBENCH_REVISION,
    "--model-revision", MODEL_REVISION,
    "--herbench-base-vlm", str(EVAL_DIR / "base_vlm.py"),
    "--herbench-base-sha256", HERBENCH_BASE_SHA256,
    "--query-method", QUERY_METHOD,
    "--max-queries", str(MAX_QUERIES),
    "--herbench-config", "lite_v2",
]
if EXAMPLE_IDS:
    common += ["--example-ids", EXAMPLE_IDS]
SURVEY = VIZ_DIR / "survey_manifest.json"

def eval_env():
    env = os.environ.copy()
    env.pop("PYTHONPATH", None)
    env.pop("PYTHONHOME", None)
    env["PYTHONUNBUFFERED"] = "1"
    env["PYTHONNOUSERSITE"] = "1"
    env["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
    return env

def _pid_alive(pid):
    try:
        os.kill(pid, 0)
        return True
    except OSError:
        return False

def _read_job():
    if not JOB_FILE.exists():
        return None
    try:
        job = json.loads(JOB_FILE.read_text(encoding="utf-8"))
        pid = int(job["pid"])
    except Exception:
        return None
    if not _pid_alive(pid):
        JOB_FILE.unlink(missing_ok=True)
        return None
    return job

def _tail_log(log_path, last_size):
    try:
        data = log_path.read_bytes()
    except OSError:
        return last_size
    if len(data) > last_size:
        print(data[last_size:].decode("utf-8", errors="replace"), end="", flush=True)
        return len(data)
    return last_size

def _kill_pg(pid):
    for sig in (signal.SIGTERM, signal.SIGKILL):
        try:
            os.killpg(pid, sig)
        except OSError:
            try:
                os.kill(pid, sig)
            except OSError:
                return
        if sig == signal.SIGTERM:
            time.sleep(2)

def _log_has_done(log_path, label):
    try:
        return f"{label} OK" in log_path.read_text(encoding="utf-8", errors="replace")
    except OSError:
        return False

def _wait_job(job):
    pid = int(job["pid"])
    log_path = Path(job["log"])
    exit_path = Path(job["exit"])
    label = str(job.get("label") or "")
    last_size = 0
    silent_s = 0
    print(f"Waiting pid={pid} label={label} log={log_path}", flush=True)
    try:
        while True:
            prev = last_size
            last_size = _tail_log(log_path, last_size)
            if last_size == prev:
                silent_s += 15
                print(f"[{time.strftime('%H:%M:%S')}] still running pid={pid} log={last_size}B silent={silent_s}s", flush=True)
            else:
                silent_s = 0
            if _log_has_done(log_path, label) and (not _pid_alive(pid) or silent_s >= 45):
                if _pid_alive(pid):
                    print(f"{label} log already done; killing pid={pid}.", flush=True)
                    _kill_pg(pid)
                    time.sleep(1)
                if not exit_path.exists():
                    exit_path.write_text("0\n", encoding="utf-8")
                break
            if not _pid_alive(pid):
                time.sleep(1.0)
                last_size = _tail_log(log_path, last_size)
                break
            if silent_s >= 720:
                print("WARN: log silent >= 12 min. Worker may be stuck.", flush=True)
                silent_s = 0
            time.sleep(15)
    except KeyboardInterrupt:
        print(f"\nCell interrupted — {label} vẫn chạy pid={pid}. Chạy lại cell này để reattach. Log: {log_path}", flush=True)
        return False
    JOB_FILE.unlink(missing_ok=True)
    code = 1
    if exit_path.exists():
        try:
            code = int(exit_path.read_text(encoding="utf-8").strip() or "1")
        except ValueError:
            code = 1
    elif _log_has_done(log_path, label):
        code = 0
    if code != 0:
        raise RuntimeError(f"{label} exited {code}. Xem {log_path}.")
    return True

def run_job(cmd, label):
    live = _read_job()
    if live is not None:
        print(f"Job đang chạy ({live.get('label')} pid={live['pid']}) — reattach, không start mới.", flush=True)
        return _wait_job(live)
    log_path = ROOT / f"frame_viz_{label.lower()}.log"
    exit_path = ROOT / f"frame_viz_{label.lower()}.exit"
    exit_path.unlink(missing_ok=True)
    print("$", " ".join(cmd), flush=True)
    inner = " ".join(shlex.quote(c) for c in cmd)
    script = f"{inner} > {shlex.quote(str(log_path))} 2>&1; echo $? > {shlex.quote(str(exit_path))}"
    proc = subprocess.Popen(
        ["bash", "-c", script],
        cwd="/content",
        env=eval_env(),
        start_new_session=True,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )
    job = {"pid": proc.pid, "label": label, "log": str(log_path), "exit": str(exit_path)}
    JOB_FILE.write_text(json.dumps(job), encoding="utf-8")
    print(f"started pid={proc.pid} (Stop cell không giết job)", flush=True)
    return _wait_job(job)

def shell_chain(parts, label):
    body = " && ".join(" ".join(shlex.quote(c) for c in part) for part in parts)
    return ["bash", "-c", body + f" && echo {label} OK"]

if _read_job() is None:
    survey_cmd = [str(PYTHON), "-u", str(RUNNER), "survey", *common]
    print("$", " ".join(survey_cmd), flush=True)
    subprocess.run(survey_cmd, check=True, env=eval_env(), cwd="/content")
if not EXAMPLE_IDS:
    if not SURVEY.is_file():
        raise RuntimeError(f"Survey did not write {SURVEY}")
    common += ["--selection-manifest", str(SURVEY)]
    print({"survey_manifest": str(SURVEY), "BATCH_N_ignored": BATCH_N}, flush=True)

live = _read_job()
if live is not None:
    ok = _wait_job(live)
    if not ok:
        print("Re-run cell 8 để reattach.")
    elif live.get("label") == "SMOKE" and RUN_BATCH:
        batch = shell_chain([
            [str(PYTHON), "-u", str(RUNNER), "collect", *common],
            [str(PYTHON), "-u", str(RUNNER), "render", "--root", "/content", "--output-dir", str(VIZ_DIR)],
        ], "BATCH")
        run_job(batch, "BATCH")
else:
    smoke_ok = True
    if SMOKE_FIRST:
        smoke_ok = run_job([str(PYTHON), "-u", str(RUNNER), "smoke", *common, "--limit", "1"], "SMOKE")
    if smoke_ok and RUN_BATCH:
        batch = shell_chain([
            [str(PYTHON), "-u", str(RUNNER), "collect", *common],
            [str(PYTHON), "-u", str(RUNNER), "render", "--root", "/content", "--output-dir", str(VIZ_DIR)],
        ], "BATCH")
        run_job(batch, "BATCH")
    elif not RUN_BATCH:
        print("RUN_BATCH=False — chỉ smoke.")


## 9. Index, contact sheet, paper groups

Contact sheet xếp theo role của prediction vừa chạy. `paper_groups.csv` là danh sách để chọn hình sau. `paper_selection.json` vẫn để trống. Không suy ra GPU smoke từ CPU checks.


In [ ]:
# 9. Show the index and English captions written by render.
from __future__ import annotations

import json
from pathlib import Path

VIZ_DIR = Path("/content/outputs_frameweave_v6_visualizations")
index = VIZ_DIR / "examples_index.csv"
groups = VIZ_DIR / "paper_groups.csv"
selection = VIZ_DIR / "paper_selection.json"
report = VIZ_DIR / "validation_report.json"
captions = VIZ_DIR / "captions_en.md"
if report.is_file():
    payload = json.loads(report.read_text(encoding="utf-8"))
    print("gpu_smoke:", payload.get("gpu_smoke"))
    print("gpu_collect:", payload.get("gpu_collect", {}).get("status"))
else:
    print("No validation_report.json yet.")
if index.is_file():
    print(index.read_text(encoding="utf-8"))
else:
    print("No examples_index.csv yet. Render has not run.")
if groups.is_file():
    print(groups.read_text(encoding="utf-8"))
else:
    print("No paper_groups.csv yet.")
if selection.is_file():
    print(selection.read_text(encoding="utf-8"))
if captions.is_file():
    print(captions.read_text(encoding="utf-8")[:4000])
print("contact sheet:", VIZ_DIR / "contact_sheet.png")


## 10. Optional read-source intervention

Mặc định tắt. Khi bật, chặn receiver khác đọc một sender rồi chạy lại forward. Không xóa ảnh, không đưa gold vào question encoding. Margin chỉ là diagnostic của vài câu.


In [ ]:
# 10. Intervention stays off unless RUN_INTERVENTION is True.
from __future__ import annotations

import os
import subprocess
from pathlib import Path

def _from_cell1(name, default):
    try:
        ip = get_ipython()
    except NameError:
        ip = None
    ns = ip.user_ns if ip is not None else {}
    return ns[name] if name in ns else default

RUN_INTERVENTION = bool(_from_cell1("RUN_INTERVENTION", False))
if not RUN_INTERVENTION:
    print("RUN_INTERVENTION=False — không chạy read-source intervention.")
else:
    ROOT = Path("/content")
    PYTHON = ROOT / "ovis_env" / "bin" / "python"
    RUNNER = ROOT / "run_frame_visualizations.py"
    VIZ_DIR = ROOT / "outputs_frameweave_v6_visualizations"
    CKPT = CKPT_DIR if "CKPT_DIR" in dir() else (ROOT / "outputs_reljoin_v6_ovis" / "checkpoints" / "best_external_val")
    cmd = [
        str(PYTHON), "-u", str(RUNNER), "intervene",
        "--root", "/content",
        "--output-dir", str(VIZ_DIR),
        "--model-dir", str(ROOT / "checkpoints" / "ovis2_5_9b"),
        "--adapter-checkpoint", str(Path(CKPT) / "Hybrid.pt"),
        "--herbench-root", str(ROOT / "HERBench_official"),
        "--videos-root", str(ROOT / "HERBench_official" / "videos"),
        "--herbench-revision", "30b42e97180f2c7c0fdc3f2aed412ae04f0366ca",
        "--model-revision", "5a1f9b995c98ae5b85a28ec89d88d74295be0ea7",
        "--herbench-base-vlm", str(ROOT / "herbench_eval_pins" / "base_vlm.py"),
        "--herbench-base-sha256", "6104884c366906d5c223cc9727c3bad778dcdf98c881c4b17b556c20e940af97",
        "--herbench-config", "lite_v2",
    ]
    example_ids = str(_from_cell1("EXAMPLE_IDS", "") or "")
    if example_ids:
        cmd += ["--example-ids", example_ids]
    env = os.environ.copy()
    env.pop("PYTHONPATH", None)
    env.pop("PYTHONHOME", None)
    env["PYTHONNOUSERSITE"] = "1"
    print("$", " ".join(cmd), flush=True)
    subprocess.run(cmd, check=True, env=env, cwd="/content")


## 11. Copy artifact sang Drive


In [ ]:
# 11. Persist visualization artifacts to Drive. Does not touch eval CSVs.
from __future__ import annotations

import shutil
from pathlib import Path

src = Path("/content/outputs_frameweave_v6_visualizations")
parent = Path("/content/drive/MyDrive/reljoin_v6_ovis_outputs")
if not parent.is_dir():
    print("Drive folder chưa mount. Bỏ qua copy.")
else:
    dest = parent / "frameweave_v6_visualizations"
    if dest.exists():
        shutil.rmtree(dest)
    shutil.copytree(src, dest)
    print("Copied to", dest)


In [ ]:
# 12. Zip the visualization folder for download.
from __future__ import annotations

import shutil
from pathlib import Path

src = Path("/content/outputs_frameweave_v6_visualizations")
zip_path = Path("/content/frameweave_v6_visualizations")
if src.is_dir():
    archive = shutil.make_archive(str(zip_path), "zip", root_dir=src)
    print(archive)
    try:
        from google.colab import files
        files.download(archive)
    except Exception as exc:
        print("Download skipped:", exc)
else:
    print("Nothing to zip yet:", src)
